# Data preparation

Shared loading and cleaning for Parts A and B.

In [1]:
from pathlib import Path
import json
import urllib.request
import zipfile

import numpy as np
import pandas as pd

DATA = Path("data")
NARR_DIR = DATA / "narratives"
RELIEF = "Closed with monetary relief"
BASE_COLS = ["Complaint ID", "Date received", "Product", "Company response to consumer"]
CAT_COLS = ["product_group", "Sub-product", "Issue", "Sub-issue", "Company", "State", "Tags"]

### Product groups

Map the older and newer product names to the same groups.

In [2]:
PRODUCT_GROUP = {
    "Credit reporting": "Credit reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Debt collection": "Debt collection",
    "Bank account or service": "Checking/savings",
    "Checking or savings account": "Checking/savings",
    "Credit card": "Credit/prepaid card",
    "Prepaid card": "Credit/prepaid card",
    "Credit card or prepaid card": "Credit/prepaid card",
    "Mortgage": "Mortgage",
    "Student loan": "Student loan",
    "Consumer Loan": "Vehicle/consumer loan",
    "Vehicle loan or lease": "Vehicle/consumer loan",
    "Payday loan": "Payday/personal loan",
    "Payday loan, title loan, or personal loan": "Payday/personal loan",
    "Payday loan, title loan, personal loan, or advance loan": "Payday/personal loan",
    "Money transfers": "Money transfer",
    "Virtual currency": "Money transfer",
    "Money transfer, virtual currency, or money service": "Money transfer",
    "Other financial service": "Other",
    "Debt or credit management": "Other",
}

### Loading and cleaning

In [3]:
def read_exports(columns=(), keep=None):
    """Load closed complaints, with optional columns and row filtering."""
    cols = list(dict.fromkeys(BASE_COLS + list(columns)))
    parts = []
    for path in sorted(NARR_DIR.glob("*.parquet")):
        d = pd.read_parquet(path, columns=cols)
        d = d[d["Company response to consumer"].str.startswith("Closed", na=False)]
        if keep is not None:
            d = d[keep(d)]
        parts.append(d)

    if not parts:
        raise FileNotFoundError("No narrative files found in data/narratives.")

    df = pd.concat(parts, ignore_index=True)
    df["year"] = df["Date received"].str[:4].astype(int)
    df["product_group"] = df["Product"].map(PRODUCT_GROUP)
    df["y"] = (df["Company response to consumer"] == RELIEF).astype(int)
    return df

In [4]:
def clean(text):
    """Remove redaction masks and export artifacts; keep mentions of money."""
    return (text.str.replace(r"^b['\"]|\\[nrt]", " ", regex=True)   # bytes-repr exports (2025+)
                .str.replace(r"\{\$[\d,.]+\}", " dollaramount ", regex=True)
                .str.replace(r"X{2,}|(?:year|scrub)>", " ", regex=True)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip())

### Archive files

In [5]:
def download():
    links = json.loads((DATA / "sources.json").read_text())["archives"]
    folder = DATA / "archive"
    folder.mkdir(exist_ok=True)

    for url in links:
        out = folder / url.rsplit("/", 1)[-1]
        if out.exists() and zipfile.is_zipfile(out):
            continue
        print(out.name)
        partial = out.with_suffix(".zip.partial")
        urllib.request.urlretrieve(url, partial)
        if not zipfile.is_zipfile(partial):
            raise ValueError(f"Not a ZIP file: {url}")
        partial.replace(out)

In [6]:
def extract():
    NARR_DIR.mkdir(exist_ok=True)
    for archive in sorted((DATA / "archive").glob("CCDB_Export_*.zip")):
        out = NARR_DIR / (archive.stem + ".parquet")
        if out.exists():
            continue
        parts = [chunk[chunk["Consumer complaint narrative"].notna()]
                 for chunk in pd.read_csv(archive, dtype=str, chunksize=500_000)]
        df = pd.concat(parts, ignore_index=True)
        df.to_parquet(out, index=False)
        print(f"{out.name}: {len(df):,} narratives")

### Part A sample

Train on 2018 to 2022, validate on 2023, and test on 2024 to 2025.

In [7]:
def sample_a():
    out = DATA / "part_a_sample.parquet"
    if out.exists():
        return

    split = {**dict.fromkeys(range(2018, 2023), "train"), 2023: "val", 2024: "test", 2025: "test"}
    frac = {"train": 0.2, "val": 0.1, "test": 0.05}
    meta = read_exports()
    meta["split"] = meta["year"].map(split)
    ids = pd.concat([g.sample(frac=frac[s], random_state=0)
                     for s, g in meta.groupby("split")])["Complaint ID"]
    df = read_exports(CAT_COLS[1:] + ["Consumer complaint narrative"],
                      keep=lambda d: d["Complaint ID"].isin(ids))
    df["split"] = df["year"].map(split)
    df["text"] = clean(df.pop("Consumer complaint narrative"))
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    df.to_parquet(out, index=False)
    print(f"{out.name}: {len(df):,} complaints")

### Part B sample

Take up to 1,000 complaints per product group and year, then remove repeated text.

In [8]:
def sample_b():
    out = DATA / "part_b_sample.parquet"
    if out.exists():
        return

    cell = ["product_group", "year"]
    meta = read_exports()
    meta = meta[meta["year"].between(2015, 2025)]
    picked = meta.sample(frac=1, random_state=0).groupby(cell).head(1000)
    df = read_exports(["Issue", "Company", "Consumer complaint narrative"],
                      keep=lambda d: d["Complaint ID"].isin(picked["Complaint ID"]))
    df["text"] = clean(df.pop("Consumer complaint narrative"))
    key = df["text"].str.lower().str.replace(r"[^a-z]", "", regex=True)
    df = df[~key.duplicated() & key.ne("")].reset_index(drop=True)
    weight = (meta.groupby(cell).size() / df.groupby(cell).size()).rename("weight")
    df = df.join(weight, on=cell)
    df.to_parquet(out, index=False)
    print(f"{out.name}: {len(df):,} complaints")

### Sentence embeddings

Use the same MiniLM model for both samples. Keep the rows in sample order.

In [9]:
def embeddings():
    missing = [name for name in ["part_a", "part_b"]
               if not (DATA / f"{name}_embeddings.npy").exists()]
    if not missing:
        return

    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    for name in missing:
        df = pd.read_parquet(DATA / f"{name}_sample.parquet")
        emb = model.encode(df["text"].tolist(), batch_size=256,
                           normalize_embeddings=True, show_progress_bar=True)
        np.save(DATA / f"{name}_embeddings.npy", emb)
        print(f"{name}: {emb.shape}")

The cached data is already prepared. These calls recreate missing files.

In [10]:
# download()
# extract()
# sample_a()
# sample_b()
# embeddings()